# DME-Encoder Final Forecast-Pretrain Sweep - Kaggle Notebook

This notebook is the classification-oriented final sweep. It starts from the checked-in `results/dme` forecast-pretrain architecture, keeps diffusion and D3PM disabled, and tests a small set of high-signal config changes against the stored full fine-tune metrics.


In [ ]:
# Cell 1 - Setup & Install
import copy
import json
import pathlib
import pickle
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

WORKING_DIR = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").exists() else pathlib.Path.cwd()
REPO_CANDIDATES = [
    WORKING_DIR / "denoising-event-sequences",
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
]
REPO_DIR = next(
    (
        p
        for p in REPO_CANDIDATES
        if (p / "src").exists() and (p / "scripts").exists()
    ),
    REPO_CANDIDATES[0],
)
if not REPO_DIR.exists():
    raise FileNotFoundError(f"Repository not found. Checked: {REPO_CANDIDATES}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR), "--quiet"],
    check=True,
)
sys.path.insert(0, str(REPO_DIR))

required_imports = {
    "yaml": "pyyaml",
    "pyarrow": "pyarrow",
    "sklearn": "scikit-learn",
    "torchmetrics": "torchmetrics",
    "matplotlib": "matplotlib",
}
missing_packages = []
for module_name, package_name in required_imports.items():
    try:
        __import__(module_name)
    except ImportError:
        missing_packages.append(package_name)
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", *missing_packages, "--quiet"],
        check=True,
    )

import numpy as np
import pandas as pd
import torch
import yaml

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

print(f"Repo       : {REPO_DIR}")
print(f"Device     : {device}")
print(f"Mixed prec : {USE_AMP}")
print(f"PyTorch    : {torch.__version__}")


In [ ]:
# Cell 2 - Paths, Targets & Runtime Knobs
DATA_ROOT = WORKING_DIR / "data"
PROCESSED_DIR = DATA_ROOT / "processed" / "rosbank_forecast_final"
OUTPUT_DIR = WORKING_DIR / "outputs" / "forecast_final_sweep"
CONFIG_DIR = OUTPUT_DIR / "configs"
LOG_DIR = OUTPUT_DIR / "logs"
RAW_PARQUET_PATH = DATA_ROOT / "rosbank_forecast_final_raw.parquet"
RAW_DATA_PATH = DATA_ROOT / "rosbank" / "train.csv.gz"

for path in [DATA_ROOT, PROCESSED_DIR, OUTPUT_DIR, CONFIG_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FAST_DEBUG = False
REBUILD_DATA = True
RUN_FORECAST_PRETRAIN = True
RUN_FROZEN_FINETUNE = False
RUN_FULL_FINETUNE = True
RUN_CONFIG_SWEEP = True

RESULTS_DME_FULL_TARGET = {
    "accuracy": 0.7506666666666667,
    "macro_f1": 0.7478692479582353,
    "weighted_f1": 0.7507020770300645,
    "roc_auc": 0.8361589642150691,
    "pr_auc": 0.8568554466678608,
    "balanced_accuracy": 0.7479410178025535,
}
METRIC_COLUMNS = ["accuracy", "macro_f1", "weighted_f1", "roc_auc", "pr_auc", "balanced_accuracy"]
SELECTION_METRICS = ["macro_f1", "balanced_accuracy", "accuracy", "pr_auc", "roc_auc"]

NUM_EPOCHS_FORECAST_PRETRAIN = 2 if FAST_DEBUG else 100
NUM_EPOCHS_FINETUNE = 2 if FAST_DEBUG else 60
BATCH_SIZE = 32 if FAST_DEBUG else 256
SMOKE_ENTITY_LIMIT = 64 if FAST_DEBUG else 512
SMOKE_BATCH_SIZE = 4 if FAST_DEBUG else 8
PRIMARY_EXPERIMENT = "forecast_repro"


def run_logged(cmd: list[str], log_path: pathlib.Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Running:", " ".join(map(str, cmd)))
    with log_path.open("w", encoding="utf-8") as f:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            f.write(line)
            f.flush()
        return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
    print(f"Log saved: {log_path}")


def load_jsonl(path: pathlib.Path) -> list[dict]:
    if not path.exists():
        print(f"Metrics file is not available yet: {path}")
        return []
    rows = []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def show_metrics_tail(path: pathlib.Path, title: str, tail: int = 8) -> pd.DataFrame:
    rows = load_jsonl(path)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    csv_path = path.with_suffix(".csv")
    df.to_csv(csv_path, index=False)
    print(title)
    print(f"JSONL: {path}")
    print(f"CSV  : {csv_path}")
    view = df.tail(tail)
    try:
        display(view)
    except NameError:
        print(view.to_string(index=False))
    return df


def find_raw_data(default_path: pathlib.Path) -> pathlib.Path:
    if default_path.exists():
        return default_path
    kaggle_input = pathlib.Path("/kaggle/input")
    candidates = sorted(kaggle_input.glob("**/train.csv.gz")) if kaggle_input.exists() else []
    if len(candidates) == 1:
        return candidates[0]
    if candidates:
        print("Multiple train.csv.gz candidates found:")
        for candidate in candidates:
            print(" -", candidate)
    raise FileNotFoundError(f"Raw data not found at {default_path}. Set RAW_DATA_PATH to the Kaggle file.")


print(f"Processed data : {PROCESSED_DIR}")
print(f"Outputs        : {OUTPUT_DIR}")
print(f"Config dir     : {CONFIG_DIR}")
print(f"Fast debug     : {FAST_DEBUG}")
print("Target results/dme full fine-tune:")
print(pd.Series(RESULTS_DME_FULL_TARGET).to_string())


In [ ]:
# Cell 3 - Final Forecast-Pretrain Config Sweep
from src.utils.config import load_experiment_config


def deep_update(base: dict, override: dict) -> dict:
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            deep_update(base[key], value)
        else:
            base[key] = value
    return base


base_config = load_experiment_config(
    base_path=str(REPO_DIR / "configs/base.yaml"),
    ablation_path=str(REPO_DIR / "configs/ablations/A14_final_forecast_pretrain.yaml"),
)

deep_update(
    base_config,
    {
        "data": {
            "max_seq_len": 256,
            "min_seq_len": 5,
            "event_type_col": "MCC",
            "timestamp_col": "TRDATETIME",
            "target_col": "target_flag",
            "numerical_cols": ["amount"],
            "categorical_cols": ["channel_type", "trx_category", "currency"],
            "group_col": "cl_id",
            "truncation_pretrain": "sliding_window",
            "truncation_eval": "last_events",
            "amount_transform": "robust_scaler",
            "amount_cols": ["amount"],
            "robust_scale_cols": ["amount"],
            "time_transform": "none",
            "min_vocab_count": 5,
            "train_ratio": 0.70,
            "val_ratio": 0.15,
            "test_ratio": 0.15,
            "use_time_features": True,
            "time_feature_mode": "derived_numeric",
        },
        "model": {
            "event_type_emb_dim": 64,
            "cat_emb_dim": 32,
            "num_projection_dim": 64,
            "time_projection_dim": 64,
            "hidden_dim": 256,
            "num_layers": 4,
            "num_heads": 8,
            "dim_feedforward": 1024,
            "dropout": 0.10,
            "activation": "gelu",
            "max_seq_len": 256,
            "use_profile_token": True,
            "client_embedding_dim": 512,
        },
        "pooling": {"type": "multi", "components": ["profile", "gated_attention", "mean", "max", "last"]},
        "forecasting": {
            "enabled": True,
            "cut_min_ratio": 0.50,
            "cut_max_ratio": 0.80,
            "min_future_events": 2,
            "event_type_target_mode": "log_future_global_ratio",
            "count_num_buckets": 6,
            "gap_num_buckets": 6,
            "alpha_forecast": 0.20,
            "alpha_denoising": 0.20,
            "lambda_event_type_profile": 1.0,
            "lambda_count": 0.5,
            "lambda_amount": 0.5,
            "lambda_gap": 0.3,
            "lambda_cat_profile": 0.5,
            "finetune_aux_enabled": False,
        },
        "loss_calibration": {"enabled": True, "warmup_steps": 1000},
        "training": {
            "task": "binary",
            "num_classes": 2,
            "main_metrics": ["roc_auc", "pr_auc", "macro_f1"],
            "batch_size": BATCH_SIZE,
            "num_epochs_pretrain": NUM_EPOCHS_FORECAST_PRETRAIN,
            "num_epochs_finetune": NUM_EPOCHS_FINETUNE,
            "lr": 0.0003,
            "lr_encoder": 0.00003,
            "weight_decay": 0.01,
            "warmup_ratio": 0.05,
            "gradient_clip_val": 1.0,
            "mixed_precision": USE_AMP,
            "early_stopping_patience": 5,
            "log_every_n_steps": 50,
        },
    },
)

CANDIDATE_OVERRIDES = {
    # Reproduce the checked-in DME setup, but with currency explicitly included.
    "forecast_repro": {},
    # Lower/higher hybrid forecast weight. These are cheap and usually the most sensitive knob.
    "forecast_alpha015": {"forecasting": {"alpha_forecast": 0.15, "alpha_denoising": 0.15}},
    "forecast_alpha030": {"forecasting": {"alpha_forecast": 0.30, "alpha_denoising": 0.30}},
    # Slightly stronger regularization: target is better ROC/PR without losing too much F1.
    "forecast_reg015": {"model": {"dropout": 0.15}, "training": {"weight_decay": 0.02}},
    # More cautious encoder update during full fine-tune.
    "forecast_enc2e5": {"training": {"lr_encoder": 0.00002, "weight_decay": 0.015}},
}

if not RUN_CONFIG_SWEEP:
    CANDIDATE_OVERRIDES = {PRIMARY_EXPERIMENT: CANDIDATE_OVERRIDES[PRIMARY_EXPERIMENT]}

CONFIG_PATHS: dict[str, pathlib.Path] = {}
EXPERIMENT_CONFIGS: dict[str, dict] = {}
for exp_name, override in CANDIDATE_OVERRIDES.items():
    cfg = copy.deepcopy(base_config)
    deep_update(cfg, copy.deepcopy(override))
    if FAST_DEBUG:
        deep_update(cfg, {"training": {"num_epochs_pretrain": 2, "num_epochs_finetune": 2, "batch_size": BATCH_SIZE}})
    cfg_path = CONFIG_DIR / f"{exp_name}.yaml"
    with cfg_path.open("w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=False)
    CONFIG_PATHS[exp_name] = cfg_path
    EXPERIMENT_CONFIGS[exp_name] = cfg

if PRIMARY_EXPERIMENT not in EXPERIMENT_CONFIGS:
    PRIMARY_EXPERIMENT = next(iter(EXPERIMENT_CONFIGS))
config = EXPERIMENT_CONFIGS[PRIMARY_EXPERIMENT]
PRIMARY_CONFIG_PATH = CONFIG_PATHS[PRIMARY_EXPERIMENT]

print(f"Primary experiment: {PRIMARY_EXPERIMENT}")
print(f"Primary config    : {PRIMARY_CONFIG_PATH}")
print(yaml.safe_dump({name: {"config": str(path)} for name, path in CONFIG_PATHS.items()}, sort_keys=False))
print(yaml.safe_dump({"data": config["data"], "model": config["model"], "forecasting": config["forecasting"], "training": config["training"]}, sort_keys=False))


In [ ]:
# Cell 4 - Data Loading & Preprocessing Artifacts
from src.data.preprocessing import EventPreprocessor
from src.data.splits import make_entity_splits


def load_raw_rosbank(path: pathlib.Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    ts_col = config["data"]["timestamp_col"]
    df[ts_col] = pd.to_datetime(df[ts_col], format="%d%b%y:%H:%M:%S", errors="coerce")
    if df[ts_col].isna().any():
        df[ts_col] = pd.to_datetime(df[ts_col], errors="coerce")
    if df[ts_col].isna().any():
        bad = int(df[ts_col].isna().sum())
        raise ValueError(f"Failed to parse {bad} timestamps in {ts_col}")
    return df


def write_raw_training_artifacts(
    raw_df: pd.DataFrame,
    splits: dict | None = None,
    prep: EventPreprocessor | None = None,
) -> tuple[pd.DataFrame, dict, EventPreprocessor]:
    data_cfg = config["data"]
    entity_col = data_cfg["group_col"]
    timestamp_col = data_cfg["timestamp_col"]
    target_col = data_cfg["target_col"]
    seed = int(config.get("seed", {}).get("global_seed", 42))

    df_sorted = raw_df.sort_values([entity_col, timestamp_col], kind="stable").reset_index(drop=True)
    helper = EventPreprocessor(config)
    df_sorted["time_delta"] = helper.compute_time_delta(df_sorted, entity_col, timestamp_col)

    if splits is None:
        splits = make_entity_splits(
            df_sorted,
            entity_col=entity_col,
            target_col=target_col,
            train_ratio=float(data_cfg.get("train_ratio", 0.70)),
            val_ratio=float(data_cfg.get("val_ratio", 0.15)),
            test_ratio=float(data_cfg.get("test_ratio", 0.15)),
            seed=seed,
            stratify=True,
        )
    if prep is None:
        prep = EventPreprocessor(config)
        prep.fit(df_sorted, splits["train"])

    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    df_sorted.to_parquet(PROCESSED_DIR / "events.parquet", index=False)
    with (PROCESSED_DIR / "preprocessor.pkl").open("wb") as f:
        pickle.dump(prep, f, protocol=pickle.HIGHEST_PROTOCOL)
    with (PROCESSED_DIR / "splits.json").open("w") as f:
        json.dump(splits, f, indent=2)
    return df_sorted, splits, prep


artifacts_exist = all(
    (PROCESSED_DIR / name).exists()
    for name in ["events.parquet", "splits.json", "preprocessor.pkl"]
)

if artifacts_exist and not REBUILD_DATA:
    df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
    with (PROCESSED_DIR / "splits.json").open() as f:
        splits = json.load(f)
    with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
        prep = pickle.load(f)
    print("Loaded existing prepared artifacts")
else:
    RAW_DATA_PATH = find_raw_data(RAW_DATA_PATH)
    raw_df = load_raw_rosbank(RAW_DATA_PATH)
    RAW_PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw_df.to_parquet(RAW_PARQUET_PATH, index=False)
    print(f"Raw data: {raw_df.shape} from {RAW_DATA_PATH}")

    try:
        run_logged(
            [
                sys.executable,
                str(REPO_DIR / "scripts/prepare_data.py"),
                "--config",
                str(PRIMARY_CONFIG_PATH),
                "--input",
                str(RAW_PARQUET_PATH),
                "--output-dir",
                str(PROCESSED_DIR),
            ],
            LOG_DIR / "prepare_data.log",
        )
        with (PROCESSED_DIR / "splits.json").open() as f:
            splits = json.load(f)
        with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
            prep = pickle.load(f)
        df_events, splits, prep = write_raw_training_artifacts(raw_df, splits=splits, prep=prep)
        print("Prepared artifacts with prepare_data.py; restored raw events.parquet for Dataset")
    except subprocess.CalledProcessError as exc:
        print(f"prepare_data.py failed with exit code {exc.returncode}; using inline artifact builder")
        df_events, splits, prep = write_raw_training_artifacts(raw_df)

print(f"Prepared events: {df_events.shape}")
print(f"Splits: train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")


In [ ]:
# Cell 5 - Prepared Artifact Inspection
df_events = pd.read_parquet(PROCESSED_DIR / "events.parquet")
with (PROCESSED_DIR / "splits.json").open() as f:
    splits = json.load(f)
with (PROCESSED_DIR / "preprocessor.pkl").open("rb") as f:
    prep = pickle.load(f)

entity_col = config["data"]["group_col"]
target_col = config["data"]["target_col"]
event_type_col = config["data"]["event_type_col"]
seq_lens = df_events.groupby(entity_col).size()
class_counts = df_events.groupby(entity_col)[target_col].first().value_counts().sort_index()

print(f"Rows          : {len(df_events):,}")
print(f"Entities      : {df_events[entity_col].nunique():,}")
print(f"Sequence len  : min={seq_lens.min()}, p50={seq_lens.median():.0f}, max={seq_lens.max()}")
print(f"Class counts  : {dict(class_counts)}")
print(f"Event vocab   : {len(prep.vocab[event_type_col])}")
print(f"Cat vocabs    : {[len(prep.vocab[c]) for c in prep.categorical_cols]}")
print(f"Num cols      : {prep.numerical_cols}")
print(f"Cat cols      : {prep.categorical_cols}")


In [ ]:
# Cell 6 - Forecast Hybrid Smoke Test
from torch.utils.data import DataLoader

from src.corruption.pipeline import CorruptionPipeline
from src.data.collate import collate_fn
from src.data.dataset import EventSequenceDataset
from src.data.forecasting import build_forecast_stats, get_num_feature_dim
from src.models.dme_encoder import DMEEncoder
from src.training.losses import compute_forecast_loss, compute_pretraining_loss


def to_device(value, target_device):
    if isinstance(value, torch.Tensor):
        return value.to(target_device)
    if isinstance(value, dict):
        return {k: to_device(v, target_device) for k, v in value.items()}
    if isinstance(value, list):
        return [to_device(v, target_device) for v in value]
    return value


def build_vocab_info(preprocessor, cfg: dict) -> dict:
    return {
        "event_type_vocab_size": len(preprocessor.vocab[preprocessor.event_type_col]),
        "cat_vocab_sizes": [len(preprocessor.vocab[col]) for col in preprocessor.categorical_cols],
        "num_num_features": get_num_feature_dim(preprocessor, cfg),
        "num_classes": int(cfg.get("training", {}).get("num_classes", 2)),
    }


smoke_ids = splits["train"][: min(len(splits["train"]), SMOKE_ENTITY_LIMIT)]
forecast_stats = build_forecast_stats(df_events, smoke_ids, prep, config)
smoke_ds = EventSequenceDataset(
    df_events,
    smoke_ids,
    prep,
    config,
    mode="forecast",
    forecast_stats=forecast_stats,
)
assert len(smoke_ds) > 0, "No valid forecast samples in smoke subset"
smoke_loader = DataLoader(
    smoke_ds,
    batch_size=SMOKE_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)
clean_batch = to_device(next(iter(smoke_loader)), device)

vocab_info = build_vocab_info(prep, config)
model = DMEEncoder(config, vocab_info).to(device)
pipe = CorruptionPipeline(
    config=config["corruption"],
    transition_matrix=None,
    vocab_sizes={
        "event_type": vocab_info["event_type_vocab_size"],
        "cat_features": vocab_info["cat_vocab_sizes"],
    },
    time_transform=config["data"].get("time_transform", "log1p"),
)

model.train()
corrupted_batch, denoise_targets, masks = pipe(clean_batch)
masks["attention_mask"] = corrupted_batch["attention_mask"]
pretrain_outputs = model(corrupted_batch, mode="pretrain")
denoise_loss = compute_pretraining_loss(pretrain_outputs, denoise_targets, masks, config)

forecast_outputs = model(clean_batch, mode="forecast")
forecast_loss = compute_forecast_loss(forecast_outputs, clean_batch["forecast_targets"], config)
hybrid_loss = denoise_loss["total"] + float(config["forecasting"].get("alpha_forecast", config["forecasting"].get("alpha_denoising", 0.2))) * forecast_loss["total"]
hybrid_loss.backward()

print(f"Smoke batch shape       : event_type={tuple(clean_batch['event_type'].shape)}")
print(f"Denoising loss          : {denoise_loss['total'].item():.4f}")
print(f"Forecast loss           : {forecast_loss['total'].item():.4f}")
print(f"Hybrid loss             : {hybrid_loss.item():.4f}")
print(f"Forecast output keys    : {sorted(forecast_outputs.keys())}")

del model, pipe, clean_batch, corrupted_batch, pretrain_outputs, forecast_outputs
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# Cell 7 - Forecast Pretraining Sweep
FORECAST_SUMMARIES: dict[str, dict] = {}
FORECAST_METRICS: dict[str, pd.DataFrame] = {}

for exp_name, cfg_path in CONFIG_PATHS.items():
    exp_dir = OUTPUT_DIR / exp_name
    forecast_dir = exp_dir / "forecast_pretrain"
    summary_path = forecast_dir / "metrics" / f"{cfg_path.stem}_forecast_pretrain_summary.json"

    if RUN_FORECAST_PRETRAIN:
        run_logged(
            [
                sys.executable,
                str(REPO_DIR / "scripts/forecast_pretrain.py"),
                "--config",
                str(cfg_path),
                "--data-dir",
                str(PROCESSED_DIR),
                "--output-dir",
                str(forecast_dir),
            ],
            LOG_DIR / f"{exp_name}_forecast_pretrain.log",
        )
    else:
        print(f"RUN_FORECAST_PRETRAIN=False; expecting existing summary for {exp_name}")

    assert summary_path.exists(), f"Missing forecast summary: {summary_path}"
    summary = json.loads(summary_path.read_text())
    ckpt_path = pathlib.Path(summary["best_checkpoint"])
    assert ckpt_path.exists(), f"Missing checkpoint: {ckpt_path}"
    FORECAST_SUMMARIES[exp_name] = {**summary, "summary_path": str(summary_path), "checkpoint": str(ckpt_path)}

    print(f"[{exp_name}] summary   : {summary_path}")
    print(f"[{exp_name}] checkpoint: {ckpt_path}")
    metrics_jsonl = forecast_dir / "logs" / f"{cfg_path.stem}_forecast_pretrain" / "metrics.jsonl"
    FORECAST_METRICS[exp_name] = show_metrics_tail(metrics_jsonl, f"{exp_name} forecast pretrain metrics")


In [ ]:
# Cell 8 - Fine-tuning Sweep
FINETUNE_ROWS = []
FINETUNE_TRAIN_METRICS: dict[str, dict[str, pd.DataFrame]] = {}

for exp_name, cfg_path in CONFIG_PATHS.items():
    exp_dir = OUTPUT_DIR / exp_name
    frozen_dir = exp_dir / "finetune_frozen"
    full_dir = exp_dir / "finetune_full"
    ckpt_path = pathlib.Path(FORECAST_SUMMARIES[exp_name]["checkpoint"])

    if RUN_FROZEN_FINETUNE:
        run_logged(
            [
                sys.executable,
                str(REPO_DIR / "scripts/finetune.py"),
                "--config",
                str(cfg_path),
                "--pretrained-checkpoint",
                str(ckpt_path),
                "--data-dir",
                str(PROCESSED_DIR),
                "--output-dir",
                str(frozen_dir),
                "--frozen-encoder",
            ],
            LOG_DIR / f"{exp_name}_finetune_frozen.log",
        )
    else:
        print(f"RUN_FROZEN_FINETUNE=False; skipping frozen encoder fine-tune for {exp_name}")

    if RUN_FULL_FINETUNE:
        run_logged(
            [
                sys.executable,
                str(REPO_DIR / "scripts/finetune.py"),
                "--config",
                str(cfg_path),
                "--pretrained-checkpoint",
                str(ckpt_path),
                "--data-dir",
                str(PROCESSED_DIR),
                "--output-dir",
                str(full_dir),
            ],
            LOG_DIR / f"{exp_name}_finetune_full.log",
        )
    else:
        print(f"RUN_FULL_FINETUNE=False; skipping full fine-tune for {exp_name}")

    frozen_metrics_path = frozen_dir / "metrics" / f"{cfg_path.stem}_finetune_metrics.json"
    full_metrics_path = full_dir / "metrics" / f"{cfg_path.stem}_finetune_metrics.json"
    if frozen_metrics_path.exists():
        FINETUNE_ROWS.append({"experiment": exp_name, "pipeline": "frozen_encoder", **json.loads(frozen_metrics_path.read_text()).get("test_metrics", {})})
    if full_metrics_path.exists():
        FINETUNE_ROWS.append({"experiment": exp_name, "pipeline": "full_finetune", **json.loads(full_metrics_path.read_text()).get("test_metrics", {})})

    FINETUNE_TRAIN_METRICS[exp_name] = {}
    frozen_jsonl = frozen_dir / "logs" / f"{cfg_path.stem}_finetune" / "metrics.jsonl"
    full_jsonl = full_dir / "logs" / f"{cfg_path.stem}_finetune" / "metrics.jsonl"
    if RUN_FROZEN_FINETUNE:
        FINETUNE_TRAIN_METRICS[exp_name]["frozen"] = show_metrics_tail(frozen_jsonl, f"{exp_name} frozen encoder fine-tune train/val metrics")
    if RUN_FULL_FINETUNE:
        FINETUNE_TRAIN_METRICS[exp_name]["full"] = show_metrics_tail(full_jsonl, f"{exp_name} full fine-tune train/val metrics")

finetune_metrics_df = pd.DataFrame(FINETUNE_ROWS)
FINETUNE_RESULTS_CSV = OUTPUT_DIR / "final_forecast_pretrain_finetune_metrics.csv"
finetune_metrics_df.to_csv(FINETUNE_RESULTS_CSV, index=False)
print(f"Saved fine-tune metrics: {FINETUNE_RESULTS_CSV}")
try:
    display(finetune_metrics_df)
except NameError:
    print(finetune_metrics_df.to_string(index=False))


In [ ]:
# Cell 9 - Compare Against Checked-In Results
if finetune_metrics_df.empty:
    comparison_df = pd.DataFrame()
    BEST_EXPERIMENT = PRIMARY_EXPERIMENT
    print("No fine-tune metrics available yet; using primary experiment as fallback.")
else:
    full_df = finetune_metrics_df[finetune_metrics_df["pipeline"] == "full_finetune"].copy()
    if full_df.empty:
        full_df = finetune_metrics_df.copy()
    for metric in METRIC_COLUMNS:
        target_value = RESULTS_DME_FULL_TARGET[metric]
        full_df[f"delta_{metric}"] = full_df[metric] - target_value
        full_df[f"beats_{metric}"] = full_df[f"delta_{metric}"] > 0
    full_df["beats_metric_count"] = full_df[[f"beats_{m}" for m in METRIC_COLUMNS]].sum(axis=1)
    full_df["selection_score"] = sum(full_df[f"delta_{m}"] for m in SELECTION_METRICS) / len(SELECTION_METRICS)
    comparison_df = full_df.sort_values(
        ["beats_metric_count", "selection_score", "macro_f1"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    BEST_EXPERIMENT = str(comparison_df.loc[0, "experiment"])

COMPARISON_CSV = OUTPUT_DIR / "final_forecast_pretrain_vs_checked_in_results.csv"
comparison_df.to_csv(COMPARISON_CSV, index=False)
print(f"Best experiment by comparison score: {BEST_EXPERIMENT}")
print(f"Comparison CSV: {COMPARISON_CSV}")
try:
    display(comparison_df)
except NameError:
    print(comparison_df.to_string(index=False))


In [ ]:
# Cell 10 - Artifact Summary
summary_rows = []
for exp_name, summary in FORECAST_SUMMARIES.items():
    row = {
        "experiment": exp_name,
        "config": str(CONFIG_PATHS[exp_name]),
        "checkpoint": summary.get("checkpoint"),
        "forecast_stats": summary.get("forecast_stats"),
    }
    if exp_name in FORECAST_METRICS and not FORECAST_METRICS[exp_name].empty:
        last_val = FORECAST_METRICS[exp_name].dropna(subset=["epoch"], how="all").tail(1)
        if not last_val.empty:
            for col in last_val.columns:
                if col.startswith("val/"):
                    row[col] = last_val.iloc[0][col]
    summary_rows.append(row)
summary_df = pd.DataFrame(summary_rows)
if not comparison_df.empty:
    keep_cols = ["experiment", "beats_metric_count", "selection_score"] + [f"delta_{m}" for m in METRIC_COLUMNS]
    summary_df = summary_df.merge(comparison_df[keep_cols], on="experiment", how="left")

SUMMARY_CSV = OUTPUT_DIR / "final_forecast_pretrain_experiment_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)
try:
    display(summary_df)
except NameError:
    print(summary_df.to_string(index=False))

summary_payload = {
    "final_architecture": "DME encoder + hybrid denoising/forecast pretraining; diffusion and D3PM disabled for classification",
    "target_results_dme_full": RESULTS_DME_FULL_TARGET,
    "best_experiment": globals().get("BEST_EXPERIMENT", PRIMARY_EXPERIMENT),
    "config_dir": str(CONFIG_DIR),
    "processed_data": str(PROCESSED_DIR),
    "experiments": {
        exp_name: {
            "config": str(CONFIG_PATHS[exp_name]),
            "forecast_summary": FORECAST_SUMMARIES.get(exp_name, {}).get("summary_path"),
            "checkpoint": FORECAST_SUMMARIES.get(exp_name, {}).get("checkpoint"),
            "forecast_stats": FORECAST_SUMMARIES.get(exp_name, {}).get("forecast_stats"),
            "output_dir": str(OUTPUT_DIR / exp_name),
        }
        for exp_name in CONFIG_PATHS
    },
    "finetune_results_csv": str(FINETUNE_RESULTS_CSV),
    "comparison_csv": str(COMPARISON_CSV),
    "summary_csv": str(SUMMARY_CSV),
    "logs": str(LOG_DIR),
}
RESULTS_JSON_PATH = OUTPUT_DIR / "final_forecast_pretrain_pipeline_results.json"
RESULTS_JSON_PATH.write_text(json.dumps(summary_payload, indent=2))

print("Final forecast-pretrain artifacts")
for key, value in summary_payload.items():
    if isinstance(value, dict):
        print(f"{key:30s}: {len(value)} entries")
    else:
        print(f"{key:30s}: {value}")
print(f"Summary JSON: {RESULTS_JSON_PATH}")
